# 01 Quickstart

From an audio file to a named feature vector: `load_audio`, a config, `extract_all_features`, `flatten_features`. Then versioned presets, the feature catalogue (including when a feature is undefined and its torch counterpart) and config validation.

Runs on a synthesized clip unless `ESC_50_dataset`, `UrbanSound8K_dataset` or `dataset_nonspeech7k` points to a local dataset (see `README.md`).

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import soundfile as sf

import afx_audio as afx
import demo_data

print('afx_audio', afx.__version__)

## Load audio

`load_audio(path, sr=...)` returns a mono float32 signal resampled to `sr` (`sr=None` keeps the file rate). `offset` and `duration` in seconds read only part of the file.

In [ ]:
tmp = Path(tempfile.mkdtemp(prefix='afx_quickstart_'))
files = demo_data.find_audio(max_files=1)
if files:
    path = files[0]
else:
    # 3 s at 44.1 kHz: a harmonic tone, then a sweep, then an AM/FM tone
    clip = np.concatenate([demo_data.synth_clip(k, 1.0, 44100, seed=i)
                           for i, k in enumerate(['tone', 'chirp', 'am_fm'])])
    path = tmp / 'example.wav'
    sf.write(path, clip, 44100)

info = sf.info(path)
print(f'{Path(path).name}: {info.samplerate} Hz, {info.channels} ch, {info.duration:.2f} s')

signal, sr = afx.load_audio(path, sr=16000)
excerpt, _ = afx.load_audio(path, sr=16000, offset=1.0, duration=1.0)
print('full   :', signal.shape, signal.dtype, sr)
print('1 s at offset 1 s:', excerpt.shape)

## One extractor

Extractors live in `afx_audio.extractors`. They take `(signal, sr)` positionally and every other parameter by keyword, and return `{name: per-frame array}`.

In [ ]:
from afx_audio.extractors import cepstral_features

out = cepstral_features.extract_mfcc(signal, sr, n_mfcc=13, frame_size=1024, hop_length=512)
print('mfcc:', out['mfcc'].shape, '(n_mfcc, n_frames)')

try:
    cepstral_features.extract_mfcc(signal, sr, 13)
except TypeError as e:
    print('TypeError:', e)

## A config for many features

A config maps feature names to extractor parameters. `extract_all_features` runs them and aggregates each over time (`'mean'` by default), so every feature becomes a fixed-size array.

In [ ]:
config = {
    'sample_rate': 16000,
    'features': {
        'mfcc': {'n_mfcc': 13, 'frame_size': 1024, 'hop_length': 512},
        'spectral_centroid': {'frame_size': 1024, 'hop_length': 512},
        'spectral_flatness': {'frame_size': 1024, 'hop_length': 512},
        'zcr': {'frame_size': 1024, 'hop_length': 512},
        'pitch': {'frame_size': 1024, 'hop_length': 512},
    },
    'aggregation': 'mean',
}
features = afx.extract_all_features(signal, sr, config)
for name, value in features.items():
    print(f'{name:18s} {value.shape}')

`flatten_features` gives the column names and the vector in the same order. Multi-value features expand to `name_0, name_1, ...`; the CSV/Parquet batch output and the scikit-learn transformer use the same names.

In [ ]:
names, vector = afx.flatten_features(features)
print(len(names), 'columns:', names[:3], '...', names[-3:])
print(vector.shape, vector.dtype)

A list of names is shorthand for default parameters, and `sample_rate` may be left out when `sr` is passed explicitly.

In [ ]:
quick = afx.extract_all_features(signal, sr, {'features': ['rms_energy', 'spectral_rolloff']})
print({k: np.round(v, 4) for k, v in quick.items()})

## Presets

Curated configs for common domains. A preset id is `name@version`: version N is a frozen file that spells out every parameter, so `speech@1` gives the same features in every AFX release that ships it. A bare name (`'speech'`) means the latest version; use the id in code you want to cite or rerun.

In [ ]:
for pid in afx.list_presets(versions=True):
    info = afx.preset_info(pid)
    print(f"{pid:15s} {info['title']:34s} {info['sample_rate']} Hz, {len(info['features'])} features")

print('\nspeech ->', afx.presets.resolve_preset('speech'))
speech = afx.load_preset('speech@1')
print('speech@1:', speech['preset'], speech['aggregation'], 'aggregation;', ', '.join(speech['features']))
names, vector = afx.flatten_features(afx.extract_all_features(signal, sr, speech))
print('vector length:', vector.size)

## What is available

`list_features(group)` lists names; `feature_info(name)` gives the group, units, rows per frame, parameters with defaults and a one-line definition. Its `undefined` entry says when a frame is NaN (None if the feature is defined on every frame, see 02), and `torch` names the `afx_audio.torch.functional` counterpart with the rtol and atol of its float64 parity test against NumPy (None without one, see 06).

In [ ]:
from afx_audio.registry import GROUPS

for group in GROUPS[:-1]:
    print(f'{group:18s} {afx.list_features(group)}')

In [ ]:
afx.feature_info('spectral_rolloff')

In [ ]:
for name in ('spectral_centroid', 'spectral_entropy', 'crest_factor', 'pitch', 'hnr'):
    info = afx.feature_info(name)
    torch_fn = info['torch']['function'].rsplit('.', 1)[-1] if info['torch'] else None
    rtol = info['torch']['rtol'] if info['torch'] else None
    print(f"{name:18s} torch={str(torch_fn):18s} rtol={str(rtol):6s} undefined: {info['undefined']}")

## Config validation

`validate_config` (called by every entry point) rejects unknown keys, features and parameters before any audio is processed, and suggests the closest valid name.

In [ ]:
bad_configs = [
    {'sample_rate': 16000, 'features': ['mfccs']},
    {'sample_rate': 16000, 'features': {'mfcc': {'n_mfc': 20}}},
    {'sample_rate': 16000, 'features': ['zcr'], 'agregation': 'mean'},
    {'sample_rate': 16000, 'features': ['zcr'], 'aggregation': 'median'},
]
for cfg in bad_configs:
    try:
        afx.validate_config(cfg)
    except ValueError as e:
        print('ValueError:', e, '\n')

Next: `02_frames_and_aggregation.ipynb` keeps the per-frame arrays and shows the aggregation methods.

In [ ]:
import shutil

shutil.rmtree(tmp)